In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    PreTrainedModel,
    AutoConfig
)

from transformers.modeling_outputs import SequenceClassifierOutput
from torch import nn

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
train_df = pd.read_csv("../data/processed/train_soft_labels.csv")
val_df = pd.read_csv("../data/processed/validation_soft_labels.csv")
test_df = pd.read_csv("../data/processed/test_soft_labels.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (13832, 23)
Validation: (2964, 23)
Test: (2965, 23)


In [5]:
text_col = "text_clean"

dist_label_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

aux_label_cols = [
    "mean_insult",
    "mean_dehumanize",
    "mean_violence",
    "mean_genocide"
]

all_label_cols = dist_label_cols + aux_label_cols

train_df[[text_col] + all_label_cols].head()

,text_clean,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,mean_insult,mean_dehumanize,mean_violence,mean_genocide
0,First off you look cool as fuck! Anyway if we ...,1.000000,0.0,0.000000,3.500000,3.500000,2.500000,1.000000
1,"They'll come back in your plan, also. Plus we ...",0.333333,0.0,0.666667,3.333333,2.666667,3.666667,3.333333
2,She ugly anyways,0.500000,0.0,0.500000,3.500000,1.500000,0.500000,0.000000
3,Do they realize that a random Japanese person ...,1.000000,0.0,0.000000,1.000000,1.000000,0.000000,0.000000
4,Won't happen. Birth rates are down. Women want...,1.000000,0.0,0.000000,1.500000,1.000000,0.000000,0.000000


In [8]:
for df in [train_df, val_df, test_df]:
    for col in aux_label_cols:
        df[col + "_norm"] = df[col] / 4.0

aux_norm_cols = [col + "_norm" for col in aux_label_cols]

train_df[aux_norm_cols].describe()

,mean_insult_norm,mean_dehumanize_norm,mean_violence_norm,mean_genocide_norm
count,13832.000000,13832.000000,13832.000000,13832.000000
mean,0.642564,0.457526,0.227952,0.115104
std,0.302905,0.295978,0.274561,0.205152
min,0.000000,0.000000,0.000000,0.000000
25%,0.500000,0.250000,0.000000,0.000000
50%,0.750000,0.500000,0.125000,0.000000
75%,0.875000,0.750000,0.333333,0.166667
max,1.000000,1.000000,1.000000,1.000000


In [ ]:
#The auxiliary columns are on a 0–4 scale. Normalizing them to 0–1 for stable MSE.
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sums = df[dist_label_cols].sum(axis=1)

    print(name)
    print("Distribution label min sum:", sums.min())
    print("Distribution label max sum:", sums.max())
    print("Distribution label mean sum:", sums.mean())
    print("Auxiliary normalized min:", df[aux_norm_cols].min().min())
    print("Auxiliary normalized max:", df[aux_norm_cols].max().max())
    print()

train
Distribution label min sum: 0.9999999999999998
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0

validation
Distribution label min sum: 0.9999999999999999
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0

test
Distribution label min sum: 0.9999999999999999
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0



In [9]:
# some checks
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    sums = df[dist_label_cols].sum(axis=1)

    print(name)
    print("Distribution label min sum:", sums.min())
    print("Distribution label max sum:", sums.max())
    print("Distribution label mean sum:", sums.mean())
    print("Auxiliary normalized min:", df[aux_norm_cols].min().min())
    print("Auxiliary normalized max:", df[aux_norm_cols].max().max())
    print()

train
Distribution label min sum: 0.9999999999999998
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0

validation
Distribution label min sum: 0.9999999999999999
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0

test
Distribution label min sum: 0.9999999999999999
Distribution label max sum: 1.0
Distribution label mean sum: 1.0
Auxiliary normalized min: 0.0
Auxiliary normalized max: 1.0



In [10]:
# converting to Hugging Face Datasets
needed_cols = [text_col] + dist_label_cols + aux_norm_cols

train_dataset = Dataset.from_pandas(train_df[needed_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[needed_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[needed_cols].reset_index(drop=True))

train_dataset

Dataset({
    features: ['text_clean', 'hatespeech_0_prob', 'hatespeech_1_prob', 'hatespeech_2_prob', 'mean_insult_norm', 'mean_dehumanize_norm', 'mean_violence_norm', 'mean_genocide_norm'],
    num_rows: 13832
})

In [11]:
model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LENGTH = 128

In [12]:
def tokenize(batch):
    return tokenizer(
        batch[text_col],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [13]:
#aDDING labels
def add_labels(batch):
    dist_labels = []
    aux_labels = []

    for i in range(len(batch[dist_label_cols[0]])):
        dist_labels.append([
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ])

        aux_labels.append([
            float(batch["mean_insult_norm"][i]),
            float(batch["mean_dehumanize_norm"][i]),
            float(batch["mean_violence_norm"][i]),
            float(batch["mean_genocide_norm"][i])
        ])

    batch["dist_labels"] = dist_labels
    batch["aux_labels"] = aux_labels

    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/13832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

In [14]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "dist_labels",
    "aux_labels"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
#shared RoBERTa encoder
#distribution head = 3 logits
#auxiliary regression head = 4 continuous outputs
class RobertaMultidimensionalAuxiliary(PreTrainedModel):
    config_class = AutoConfig

    def __init__(self, config, model_name, num_dist_labels=3, num_aux_labels=4, dropout_prob=0.1):
        super().__init__(config)

        self.roberta = AutoModel.from_pretrained(model_name, config=config)

        hidden_size = config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        self.dist_head = nn.Linear(hidden_size, num_dist_labels)
        self.aux_head = nn.Linear(hidden_size, num_aux_labels)

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        dist_labels=None,
        aux_labels=None,
        **kwargs
    ):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        dist_logits = self.dist_head(pooled)
        aux_preds = torch.sigmoid(self.aux_head(pooled))

        return {
            "dist_logits": dist_logits,
            "aux_preds": aux_preds
        }

In [17]:
config = AutoConfig.from_pretrained(model_name)

model = RobertaMultidimensionalAuxiliary(
    config=config,
    model_name=model_name,
    num_dist_labels=3,
    num_aux_labels=4,
    dropout_prob=0.1
)

model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaMultidimensionalAuxiliary(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [18]:
LAMBDA_AUX = 0.3
class MultidimensionalAuxiliaryTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        dist_labels = inputs.pop("dist_labels").float()
        aux_labels = inputs.pop("aux_labels").float()

        outputs = model(**inputs)

        dist_logits = outputs["dist_logits"]
        aux_preds = outputs["aux_preds"]

        log_probs = torch.nn.functional.log_softmax(dist_logits, dim=-1)

        dist_loss = torch.nn.functional.kl_div(
            log_probs,
            dist_labels,
            reduction="batchmean"
        )

        aux_loss = torch.nn.functional.mse_loss(
            aux_preds,
            aux_labels
        )

        total_loss = dist_loss + (LAMBDA_AUX * aux_loss)

        loss_outputs = {
            "loss": total_loss,
            "dist_loss": dist_loss.detach(),
            "aux_loss": aux_loss.detach(),
            "dist_logits": dist_logits,
            "aux_preds": aux_preds
        }

        return (total_loss, loss_outputs) if return_outputs else total_loss

In [19]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Depending on Trainer version, predictions may arrive as tuple/list/dict-like.
    if isinstance(predictions, tuple):
        dist_logits = predictions[0]
        aux_preds = predictions[1]
    elif isinstance(predictions, dict):
        dist_logits = predictions["dist_logits"]
        aux_preds = predictions["aux_preds"]
    else:
        dist_logits = predictions

    #  computing metrics manually below
    # by relying on eval dataset predictions through Trainer. 
    # This function will be used only if labels are properly passed.
    return {}

In [20]:
training_args = TrainingArguments(
    output_dir="../models/roberta_multidimensional_auxiliary_1",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=8,
    weight_decay=0.01,

    logging_steps=100,
    load_best_model_at_end=False,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

In [21]:
trainer = MultidimensionalAuxiliaryTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [22]:
torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.557700,0.529270
2,0.497600,0.570132
3,0.430700,0.595377
4,0.355300,0.667985
5,0.302900,0.745187
6,0.258300,0.851079
7,0.222500,0.879448
8,0.204200,0.937806


TrainOutput(global_step=13832, training_loss=0.357926372608882, metrics={'train_runtime': 1287.752, 'train_samples_per_second': 85.93, 'train_steps_per_second': 10.741, 'total_flos': 4438035842935200.0, 'train_loss': 0.357926372608882, 'epoch': 8.0})

In [23]:
from torch.utils.data import DataLoader
def predict_multidim(model, dataset, batch_size=32):
    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        collate_fn=data_collator
    )

    all_dist_probs = []
    all_aux_preds = []
    all_dist_labels = []
    all_aux_labels = []

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        dist_labels = batch.pop("dist_labels").float()
        aux_labels = batch.pop("aux_labels").float()

        with torch.no_grad():
            outputs = model(**batch)

        dist_probs = torch.nn.functional.softmax(
            outputs["dist_logits"],
            dim=-1
        )

        all_dist_probs.append(dist_probs.cpu().numpy())
        all_aux_preds.append(outputs["aux_preds"].cpu().numpy())
        all_dist_labels.append(dist_labels.cpu().numpy())
        all_aux_labels.append(aux_labels.cpu().numpy())

    return {
        "dist_probs": np.vstack(all_dist_probs),
        "aux_preds": np.vstack(all_aux_preds),
        "dist_labels": np.vstack(all_dist_labels),
        "aux_labels": np.vstack(all_aux_labels)
    }

In [24]:
def evaluate_predictions(pred_dict):
    probs = pred_dict["dist_probs"]
    labels = pred_dict["dist_labels"]

    aux_preds = pred_dict["aux_preds"]
    aux_labels = pred_dict["aux_labels"]

    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    pred_hard = probs.argmax(axis=1)
    gold_hard = labels.argmax(axis=1)

    hard_accuracy = (pred_hard == gold_hard).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    aux_mse = np.mean((aux_preds - aux_labels) ** 2)
    aux_mae = np.mean(np.abs(aux_preds - aux_labels))

    per_aux_mse = {
        f"{col}_mse": np.mean((aux_preds[:, i] - aux_labels[:, i]) ** 2)
        for i, col in enumerate(aux_label_cols)
    }

    per_aux_mae = {
        f"{col}_mae": np.mean(np.abs(aux_preds[:, i] - aux_labels[:, i]))
        for i, col in enumerate(aux_label_cols)
    }

    results = {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr),
        "aux_mse": float(aux_mse),
        "aux_mae": float(aux_mae)
    }

    results.update(per_aux_mse)
    results.update(per_aux_mae)

    return results

In [25]:
val_predictions = predict_multidim(model, val_dataset, batch_size=32)
test_predictions = predict_multidim(model, test_dataset, batch_size=32)

val_results = evaluate_predictions(val_predictions)
test_results = evaluate_predictions(test_predictions)

val_results, test_results

({'kl_divergence': 0.9234347939491272,
  'js_divergence': 0.1453741192817688,
  'hard_accuracy': 0.7432523616734144,
  'entropy_correlation': 0.2136811164262195,
  'aux_mse': 0.04793446511030197,
  'aux_mae': 0.164974644780159,
  'mean_insult_mse': np.float32(0.04912248),
  'mean_dehumanize_mse': np.float32(0.060770955),
  'mean_violence_mse': np.float32(0.050108552),
  'mean_genocide_mse': np.float32(0.03173587),
  'mean_insult_mae': np.float32(0.1717036),
  'mean_dehumanize_mae': np.float32(0.1988636),
  'mean_violence_mae': np.float32(0.16817676),
  'mean_genocide_mae': np.float32(0.12115464)},
 {'kl_divergence': 0.9768348932266235,
  'js_divergence': 0.15426461398601532,
  'hard_accuracy': 0.7258010118043845,
  'entropy_correlation': 0.21455632864623006,
  'aux_mse': 0.051734719425439835,
  'aux_mae': 0.17023985087871552,
  'mean_insult_mse': np.float32(0.048620198),
  'mean_dehumanize_mse': np.float32(0.06513214),
  'mean_violence_mse': np.float32(0.057612132),
  'mean_genocide_ms

In [26]:
model_save_path = "../models/roberta_multidimensional_auxiliary_1/final_model"

os.makedirs(model_save_path, exist_ok=True)

model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("Saved model to:", model_save_path)

Saved model to: ../models/roberta_multidimensional_auxiliary_1/final_model


In [ ]:
import json
from datetime import datetime

os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_multidimensional_auxiliary_1_results.txt"
history_path = "../results/tables/roberta_multidimensional_auxiliary_1_training_history.csv"

history_df = pd.DataFrame(trainer.state.log_history)
history_df.to_csv(history_path, index=False)

train_logs = history_df[history_df["loss"].notna()] if "loss" in history_df.columns else pd.DataFrame()
eval_logs = history_df[history_df["eval_loss"].notna()] if "eval_loss" in history_df.columns else pd.DataFrame()

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA MULTIDIMENSIONAL AUXILIARY MODEL RESULTS\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {training_args.output_dir}\n")
    f.write(f"Global training steps: {trainer.state.global_step}\n\n")

    f.write("2. MODEL OBJECTIVE\n")
    f.write("-" * 80 + "\n")
    f.write("Main task: 3-class hatespeech soft-label distribution prediction\n")
    f.write("Auxiliary task: regression over selected MHS harmfulness dimensions\n")
    f.write(f"Auxiliary loss weight lambda: {LAMBDA_AUX}\n\n")

    f.write("Distribution target columns:\n")
    for col in dist_label_cols:
        f.write(f"- {col}\n")

    f.write("\nAuxiliary regression columns:\n")
    for col in aux_label_cols:
        f.write(f"- {col}\n")
    f.write("\n")

    f.write("3. CUDA / HARDWARE\n")
    f.write("-" * 80 + "\n")
    f.write(f"CUDA available: {torch.cuda.is_available()}\n")
    if torch.cuda.is_available():
        f.write(f"GPU: {torch.cuda.get_device_name(0)}\n")
        f.write(f"CUDA version: {torch.version.cuda}\n")
    f.write("\n")

    f.write("4. DATASET SIZES\n")
    f.write("-" * 80 + "\n")
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Validation rows: {len(val_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n\n")

    f.write("5. TRAINING SETUP\n")
    f.write("-" * 80 + "\n")
    f.write(f"Max sequence length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {training_args.per_device_train_batch_size}\n")
    f.write(f"Eval batch size: {training_args.per_device_eval_batch_size}\n")
    f.write(f"Epochs: {training_args.num_train_epochs}\n")
    f.write(f"Learning rate: {training_args.learning_rate}\n")
    f.write(f"Weight decay: {training_args.weight_decay}\n")
    f.write(f"FP16: {training_args.fp16}\n\n")

    f.write("6. FINAL VALIDATION RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in val_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("7. FINAL TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    for key, value in test_results.items():
        f.write(f"{key}: {value}\n")
    f.write("\n")

    f.write("8. TRAINING HISTORY SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(f"Full training history saved to: {history_path}\n")
    f.write(f"Total log entries: {len(history_df)}\n")
    f.write(f"Training loss log entries: {len(train_logs)}\n")
    f.write(f"Evaluation log entries: {len(eval_logs)}\n\n")

    if not train_logs.empty:
        f.write("Training loss summary:\n")
        f.write(train_logs["loss"].describe().to_string())
        f.write("\n\n")

        f.write("First 5 training loss logs:\n")
        f.write(train_logs[["epoch", "step", "loss"]].head().to_string(index=False))
        f.write("\n\n")

        f.write("Last 5 training loss logs:\n")
        f.write(train_logs[["epoch", "step", "loss"]].tail().to_string(index=False))
        f.write("\n\n")

    if not eval_logs.empty:
        f.write("Evaluation logs:\n")
        f.write(eval_logs.to_string(index=False))
        f.write("\n\n")

print("Saved result report:", results_path)
print("Saved training history:", history_path)

Saved result report: ../results/tables/roberta_multidimensional_auxiliary_1_results.txt
Saved training history: ../results/tables/roberta_multidimensional_auxiliary_1_training_history.csv
